# The Simpsons viewer Analytics

## Data Visualization MDS/MEI

**Author:** Alexis Vendrix

**Date:** 30/03/2026



This document contains the data processing pipeline and the analytical design process for *The Simpsons* viewers dataset.

The Streamlit global visualization is provided in a standalone python code and is available online: [Simpson web application](https://2026upcdatavisualisationsimpsons.streamlit.app/)


![Dashboard](images/DV_All.png)

### Libraries

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from sklearn.linear_model import LinearRegression

## 1. Data Processing and Cleaning

The original dataset requires cleaning:
1. **Drop irrelevant columns:** `image_url`, `video_url`, and `production_code` are removed.
2. **Date conversion:** Convert `original_air_date` to datetime format, computing year, month, weekday number, week number and full weekday name.
3. **Handle missing values:** 
   * I removed the season 28 entirely as critical data has not been collected yet. I considered that the data for that season is not truthworthy.
   * Impute missing values for `us_viewers_in_millions` in season 8 using a **Linear Regression** model predicting millions of viewers off the `number_in_season` feature. The regression is only made with the data within that season. 
4. **Standardize types:** I insure that seasons and episode numbers are integers.

### Extra:
1. **Trend calculation:** I calculate the linear regression slope of viewers across episodes for each season. This explicitly determines if each season was generally gaining or losing viewers (Used for Viewership Patterns per Season).
2. **Viewer metric distinction:** Create a `viewers_type` column to logically separate "Household Viewers" (Seasons <= 11) and "Individual Viewers" (Seasons > 11), correctly annotating a measurement methodology shift in late 90s television.


In [ ]:
def clean_data(input_path, output_path):
    df = pd.read_csv(input_path)
    
    # 1. Drop irrelevant columns
    cols_to_drop = ['image_url', 'video_url', 'production_code']
    df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)
    
    # 2. Date parsing
    df['original_air_date'] = pd.to_datetime(df['original_air_date'], errors='coerce')
    df['week_num'] = df['original_air_date'].dt.week
    df['weekday_num'] = df['original_air_date'].dt.weekday
    df['weekday_name'] = df['original_air_date'].dt.day_name()
    df['year'] = df['original_air_date'].dt.year
    df['month'] = df['original_air_date'].dt.month

    # Standardize types
    df['season'] = df['season'].astype(int)
    df['number_in_season'] = df['number_in_season'].astype(int)
    df['number_in_series'] = df['number_in_series'].astype(int)

    # 3. Handle missing values
    # Remove season 28
    df = df[df['season'] != 28]

    # Impute missing values - us_viewers_in_millions in season 8
    df_s8_train = df[df['season'] == 8].dropna(subset=['number_in_season', 'us_viewers_in_millions'])
    X_s8 = df_s8_train[['number_in_season']]
    y_s8 = df_s8_train['us_viewers_in_millions']

    model_s8 = LinearRegression()
    model_s8.fit(X_s8, y_s8)

    s8_missing_mask = (df['season'] == 8) & df['us_viewers_in_millions'].isnull()
    if s8_missing_mask.any():
        df.loc[s8_missing_mask, 'us_viewers_in_millions'] = model_s8.predict(df.loc[s8_missing_mask, ['number_in_season']])
    
    # Simple function to calculate the slope for a group of episodes with numpy
    def calculate_slope(group):
        if len(group) > 1: 
            return np.polyfit(group['number_in_season'], group['us_viewers_in_millions'], 1)[0]
        return 0

    trend_data = df.groupby('season').apply(calculate_slope).reset_index(name='trend_slope')
    df = pd.merge(df, trend_data, on='season', how='left')

    # Create viewers_type column
    df['viewers_type'] = df['season'].apply(lambda x: 'Household Viewers (Millions)' if x <= 11 else 'Individual Viewers (Millions)')
    
    df.to_csv(output_path, index=False)
    return df

    # script_dir = os.path.dirname(os.path.abspath(__file__))
    # input_csv = os.path.join(script_dir, "simpsons_episodes.csv")
    # output_csv = os.path.join(script_dir, "simpsons_episodes_clean.csv")
    # clean_data(input_csv, output_csv)

### Clean data import

In [ ]:
df = pd.read_csv('simpsons_episodes_clean.csv')

# 2. Design

## 2.1 Global thematic configuration for Streamlit
To ensure a coherent design, the dashboard's interface was globally themed using Streamlit's configuration file. Rather than hardcoding colors into individual frontend components, I used a color palette inspired by The Simpsons for different part of the design. The sidebar was designated in a yellow container (#FED41D)and the main analytical area utilizes a clean off-white (#FFFEF9).


## 2.2. Global thematic configuration for Altair
Instead of manually formatting every individual chart, the code defines and registers a custom Altair theme (simpsons_theme). This ensures visual consistency across the entire dashboard.

Custom color palette: An array of hex codes is defined, mapping directly to The Simpsons brand colors (e.g., Marge Blue #009DDC, Donut Pink #F14A9C).

Background integration: Setting the background and view stroke to "transparent" removes Altair’s default white boxes and borders. This give us a more minimalistic design.

Standardized typography: All chart titles and axis labels are globally configured to use a clean sans-serif font, specific font sizes (16pt and 18pt), and coordinated colors, guaranteeing a unified, professional aesthetic without requiring redundant code on each chart.

In [8]:

def simpsons_theme():
    simpsons_palette = ["#009DDC", "#F05E23", "#F14A9C", "#D1B271", "#4C76B6"]
    return alt.theme.ThemeConfig(
        # For a better visualisation, the transparency is only used for the final Streamlit visualisation.
        #background="transparent",
        view={"stroke": "transparent"},
        range={
            "category": simpsons_palette,
            "heatmap": ["#009DDC", "#FFFFFF", "#F14A9C"],
        },
        title={
            "font": "sans-serif",
            "color": "#F14A9C",
            "fontSize": 18,
            "anchor": "start",
        },
        axis={
            "labelColor": "#333333",
            "titleColor": "#F14A9C",
            "gridColor": "#D0D0D0",
        },
    )
alt.themes.register("simpsons", simpsons_theme)
alt.themes.enable("simpsons")


ThemeRegistry.enable('simpsons')

# 3. Implementation

Globally, I have increase the size of the font for the axis, legends, ticks,...

## 3.1. Rating row

![Rating](images/DV_Ratings.png)

Both of those graphs show the IMBD Rating. For this reason, they were placed next to each other with a y-axis representing the rating. Moreover, i'm using the same blue on purpose.

### 3.1.1. Evolution of rating (Question 1)

**Design decisions:**
- **Chart choice (Boxplot vs. Line Chart):** Simply plotting the average rating per season via a standard line chart hides a lot of information. A Boxplot was chosen because it reveals the variance. It allows the viewer to instantly see not just the median quality, but the spread—proving whether a season was consistently average, or a highly volatile mix of brilliant masterpieces and terrible flops.
- **The trendline overlay:** A pink LOESS regression line was overlaid to guide the viewer's eye. It cuts through the season-to-season noise to definitively highlight the evolution of the ratings.
- **Y-Axis zooming (domain=[4, 10]):** Because The Simpsons IMDb ratings rarely drop below 4.0, we zoomed in on the Y-axis to make the variations more visible.
- **Color strategy:** The chart uses a "Background/Foreground" separation blue/pink. The blue boxplots act as the dense background data and represent the rating, while the Pink line brings the analytical trend right to the front.


In [ ]:
# The x_var and x_title implementation is not useful for the first part of the project 
# but will be needed for sliders or selection for the second part of the project.

x_var = 'season:O'
x_title = 'Season Number'

base_boxplot_ratings = alt.Chart(df).encode(
    x=alt.X(x_var, title=x_title, axis=alt.Axis(labelAngle=0))
)

bp_ratings = base_boxplot_ratings.mark_boxplot(extent='min-max', color='#009DDC').encode(
    y=alt.Y('imdb_rating:Q', title='IMDB Rating', scale=alt.Scale(domain=[4, 10]))
)

trend_bp_ratings = base_boxplot_ratings.transform_loess(
    'season', 'imdb_rating'
).mark_line(color='#F14A9C', strokeWidth=3).encode(
    y='imdb_rating:Q'
)

chart_r = (bp_ratings + trend_bp_ratings).properties(
    height=400,
    width=800,
    title="IMDB Rating evolution over time"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    fontSize=20
)

chart_r


alt.LayerChart(...)

### 3.1.2. Correlation analysis between ratings and viewers (Question 3)

**Design Decisions:**
- **Chart Choice (Scatter Plot):** To prove whether higher viewership correlates with higher rating, a scatter plot is the standard. It plots two continuous variables explicitly against each other.
- **Overplotting mitigation:** With hundreds of episodes plotted, data points inevitably overlap. By setting circle opacity to 0.8, overlapping transparent dots multiply to create a darker hue, showing where the heaviest concentration lies.
- **The regression line:** A pink regression line serves as an immediate visual summary of the calculated 'r' value, proving the exact trajectory.
- **In-Chart annotation:** The Pearson correlation coefficient (`r`) is embedded directly into the chart's canvas to visually tie the mathematical value to the line, without asking users to read a paragraph below.
- **Tooltips:** Rich tooltips provide details for each episode to contextualize outliers.
- **Color Strategy:** The chart uses a "Background/Foreground" separation blue/pink like the previous chart. The blue points act as the dense background data and represent the rating, while the Pink line brings the analytical trend right to the front.


In [ ]:
corr_value = df['us_viewers_in_millions'].corr(df['imdb_rating'])

base_scatter = alt.Chart(df).encode(
    x=alt.X('us_viewers_in_millions:Q', title='Viewers (Millions)'),
    y=alt.Y('imdb_rating:Q', title='IMDB Rating', scale=alt.Scale(domain=[4, 10]))
)

scatter_dots = base_scatter.mark_circle(size=60, opacity=0.8, color='#009DDC').encode(
    tooltip=['title:N', 'season:O', 'number_in_season:O', 'us_viewers_in_millions:Q', 'imdb_rating:Q']
)

regression_line = base_scatter.transform_regression(
    'us_viewers_in_millions', 'imdb_rating'
).mark_line(color='#F14A9C', strokeWidth=3)

text_df = pd.DataFrame({
    'x': [df['us_viewers_in_millions'].max()],
    'y': [9.5], 
    'label': [f"r = {corr_value:.2f}"]
})

text_annotation = alt.Chart(text_df).mark_text(
    align='right',
    fontSize=20,
    fontWeight='bold',
    color='#F14A9C'
).encode(
    x='x:Q',
    y='y:Q',
    text='label:N'
)

chart_corr = (scatter_dots + regression_line + text_annotation).properties(
    height=400,
    width=800,
    title="Correlation (r) analysis between ratings and viewers"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    fontSize=20
)

chart_corr


alt.LayerChart(...)

## 3.2. Viewership row

![Viewers](images/DV_Viewers_2.png)

Both of those graphs show the number of viewers. For this reason, they were placed next to each other with a y-axis for the number of viewers. I'm using different tint of teal to represent the results.

### 3.2.1. Viewership evolution over time (Question 2)

*Note: After Season 11, the measurement metric shifted from counting the total number of households to the number of individual people.*

**Design decisions:**
- **Chart choice (Dual-Metric Boxplot):** When analyzing viewership, averages are deceiving. A single highly anticipated episode can heavily skew a season's average. Boxplots reveal the true spread, showing both baseline viewership and massive outlier events for each season.
- **Grouped LOESS Trendlines:** A locally weighted smoothing line was calculated for each viewer type independently to cut through noise and reveal the overall trend of a steep viewer decline.
- **Methodological Transparency (Color Separation):** To reflect the viewer measurement methodology shift transparently, data was grouped and mapped to a Teal color scale. The visual contrast honestly separates the two measurement eras.


In [11]:
base_boxplot_viewers = alt.Chart(df).encode(
    x=alt.X(x_var, title=x_title, axis=alt.Axis(labelAngle=0))
)

bp_viewers = base_boxplot_viewers.mark_boxplot(extent='min-max').encode(
    y=alt.Y('us_viewers_in_millions:Q', title='Viewers (Millions)'),
    color=alt.Color(
        'viewers_type:N', 
        title='Viewers Type', 
        legend=alt.Legend(orient='top-right'),
        scale=alt.Scale(
            domain=['Household Viewers (Millions)', 'Individual Viewers (Millions)'],
            range=["#39989f", "#6ccbd2"]        
        )
    )
)

trend_bp_viewers = base_boxplot_viewers.transform_loess(
    'season', 'us_viewers_in_millions', groupby=['viewers_type']
).mark_line(color='#F14A9C', strokeWidth=3).encode(
    y='us_viewers_in_millions:Q',
    detail='viewers_type:N'
)

chart_v = (bp_viewers + trend_bp_viewers).properties(
    height=400,
    width=800,
    title="Viewership evolution over time"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_legend(
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    fontSize=20
)

chart_v


alt.LayerChart(...)

### 3.2.2. Viewers by airing weekday (Question 4)

**Design Decisions:**
- **Chart choice (bar chart):** The optimal choice for comparing aggregated, discrete categorical data (Days of the Week).
- **Data filtering:** I choose to filter the dataset to the main broadcast nights (`Thursday` and `Sunday`). Other days contain rare special airings with statistically misleading averages due to their week occurence.
- **On graph data labeling:** Averages are embedded directly inside the bars using a clear font size. This visually give the core information without forcing the eyes to check the viewer axis: Thursday's historical average viewership is definitively more than double Sunday's average.


In [ ]:
main_days = ['Thursday', 'Sunday']
df_main_days = df[df['weekday_name'].isin(main_days)]

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

chart4 = alt.Chart(df_main_days).mark_bar(color="#2ec4b6").encode(
    x=alt.X('weekday_name:N', title=None, axis=alt.Axis(labelAngle=0), sort=weekday_order),
    y=alt.Y('mean(us_viewers_in_millions):Q', title='Avg Viewers (Millions)'),
    tooltip=[
        alt.Tooltip('weekday_name:N', title='Weekday'),
        alt.Tooltip('mean(us_viewers_in_millions):Q', title='Average viewers (M)', format='.2f'),
        alt.Tooltip('count():Q', title='Number of episodes') 
    ]
)

chart4_text = chart4.mark_text(
    align="center",
    baseline="bottom",
    dy=72, 
    color="#FFFFFF",
    fontSize=48,
    fontWeight="bold"
).encode(
    text=alt.Text("mean(us_viewers_in_millions):Q", format=".1f")
)

final_bar_chart = (chart4 + chart4_text).properties(
    height=400,
    width=400,
    title="Viewers by airing weekday"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    fontSize=20
)

final_bar_chart


alt.LayerChart(...)

## 3.3. Viewership Patterns per Season (Question 5)

![Rating](images/DV_Heatmap.png)

**Design Decisions:**
- **Heatmap and line chart combination:** While heatmaps are good at revealing local maximums, people may have difficulties to visually calculate a rate of change across a color gradient. Moreover, I found that the general pattern is not directly clear. To help the user, a linear regression slope column explicitly displays alongside the data, providing a definitive, quantitative answer for whether a season historically gained or lost viewers throughout its run. The trend indicator represents the average change in viewers per episode across a given season.
- **Double level of analysis:** The composite layout bridges two levels of analysis. The heatmap provides the "Micro" view (episode-to-episode volatility), while the trend column provides the "Macro" view (audience trajectory).
- **Diverging color code:**  In the trend, teal is used for positive growth, pink for decrease, allowing for quick scanning to identify seasons with strong audience retention.


In [ ]:
unique_trend_df = df[['season', 'trend_slope']].drop_duplicates(subset=['season'])

trend_chart = alt.Chart(unique_trend_df).mark_text(fontWeight='bold', align='center', fontSize=14).encode(
    y=alt.Y('season:O', sort='descending', title='Season number', axis=alt.Axis(ticks=False, labels=False)), 
    x=alt.X('dummy:O', title='Trend', axis=alt.Axis(
        labels=True,              
        labelColor='transparent', 
        ticks=True,               
        tickColor='transparent',  
        domain=False              
    )),
    text=alt.Text('trend_slope:Q', format='+.2f'), 
    color=alt.condition(
        alt.datum.trend_slope > 0,
        alt.value("#2ec4b6"), 
        alt.value("#F14A9C")  
    ),
    tooltip=[
        alt.Tooltip('trend_slope:Q', title='Total Viewers Changed', format='+.2f')
    ]
).transform_calculate(
    dummy="' '" 
).properties(
    height=alt.Step(20),
    width=80
)

chart5 = alt.Chart(df).mark_rect().encode(
    x=alt.X('number_in_season:O', title='Episode number', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('season:O', title=None, axis=alt.Axis(labels=False, ticks=False), sort='descending'),
    color=alt.Color('us_viewers_in_millions:Q', title='Viewers (M)', scale=alt.Scale(scheme='inferno', reverse=True)),
    tooltip=['season:N', 'number_in_season:Q', 'us_viewers_in_millions:Q', 'title:N']
).properties(
    height=alt.Step(20),
    width=600
)

final_heatmap = alt.hconcat(trend_chart, chart5, spacing=10).properties(
    title="Viewership Patterns per Season"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    fontSize=20
)

final_heatmap


alt.HConcatChart(...)

# 4. Final visualization

![Rating](images/DV_Final_1.png)

![Rating](images/DV_Final_2.png)

The final dashboard employs a cohesive design, utilizing a "Simpsons Yellow" sidebar with an "About section" and global KPIs. The main countainer with the visualizations is minimalistic with low data-ink ratio. A pink donut color serves as a accent color for titles and trendlines.

The layout is engineered for data storytelling. The top section utilizes a 2x2 grid to establish macro-level observations: Top left (Rating over time) > Top right (Correlation Rating/Viewers) > Bottom left (Viewer over time) > Botting right (Viewers per weekday).

The graphs on the same row are using the same tint of the same color (Blue or teal) because they represent the same type of data.

The last bottom section represent the micro analysis.  The full viewership heatmap give insights per episode. The inferno sequential color scale maps audience density. The custom "Trend" column sits next to the heatmap’s left Y-axis. This allows the user to check the rate-of-change within a season with a diverging color (Teal for positive momentum, Pink for negative) to show the mathematic trend.